# game_utils explorer

A scratchpad for `agent_code/my_agent/game_utils.py`. Every function in there is **pure** --
hand it a state dict, it returns an answer. No game loop needed.

Run the setup cell once, then poke at anything. Edit the board, move the bomb, re-run.


In [2]:
import sys, os
from pathlib import Path

# Notebooks start in their own folder; the agent code imports `settings`, which
# only resolves from the repository root.
ROOT = Path.cwd()
while not (ROOT / 'settings.py').exists():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np
import settings as s

from agent_code.my_agent.game_utils import (
    DIRECTIONS, LARGE, BLAST_LINGER, OWN_BOMB_DEADLINE,
    blast_coords, bomb_deadlines, is_lethal_at, passable, free_tiles,
    can_survive, bfs_first_step, bfs_to_safety, crates_hit, opponents_hit)

print('root:', ROOT.name)
print(f'BOMB_POWER={s.BOMB_POWER}  BOMB_TIMER={s.BOMB_TIMER}  '
      f'BLAST_LINGER={BLAST_LINGER}  OWN_BOMB_DEADLINE={OWN_BOMB_DEADLINE}  LARGE={LARGE} BOMBDEADLINE={bomb_deadlines}')


root: bomberman_rl
BOMB_POWER=3  BOMB_TIMER=4  BLAST_LINGER=1  OWN_BOMB_DEADLINE=5  LARGE=999 BOMBDEADLINE=<function bomb_deadlines at 0x10cc6b110>


## Building a board by hand

`empty_field()` gives the standard 17x17 maze: border walls plus a pillar wherever
`x` and `y` are both even. `mk()` wraps it into the dict the framework would pass to `act()`.


In [3]:
def empty_field():
    f = np.zeros((s.COLS, s.ROWS), dtype=int)
    f[0,:] = f[-1,:] = f[:,0] = f[:,-1] = -1          # border
    for x in range(1, s.COLS-1):
        for y in range(1, s.ROWS-1):
            if x % 2 == 0 and y % 2 == 0:
                f[x, y] = -1                           # pillars
    return f


def mk(field, pos, bombs=(), coins=(), others=(), bombs_left=True, expl=None, step=10):
    """A game_state dict, exactly as environment.get_state_for_agent would build it."""
    return {'round': 1, 'step': step, 'field': field,
            'bombs': list(bombs), 'coins': list(coins),
            'self': ('me', 0, bombs_left, pos), 'others': list(others),
            'explosion_map': np.zeros_like(field) if expl is None else expl,
            'user_input': None}


def draw(st, w=11, h=9, mark=()):
    """Top-left corner of the board. `mark` highlights extra tiles with *."""
    field = st['field']
    bombs = {tuple(p): t for p, t in st['bombs']}
    coins = {tuple(c) for c in st['coins']}
    others = {tuple(o[3]) for o in st['others']}
    me, mark = tuple(st['self'][3]), {tuple(m) for m in mark}
    for y in range(h):
        row = []
        for x in range(w):
            if   (x,y) == me:        c = '@'
            elif (x,y) in bombs:     c = str(bombs[(x,y)])
            elif (x,y) in others:    c = 'E'
            elif (x,y) in mark:      c = '*'
            elif (x,y) in coins:     c = 'o'
            elif st['explosion_map'][x,y] >= 1: c = '!'
            elif field[x,y] == -1:   c = '#'
            elif field[x,y] ==  1:   c = '+'
            else:                    c = '.'
            row.append(c)
        print('  ' + ' '.join(row))


f = empty_field()
st = mk(f, (1,1), coins=[(7,1)])
draw(st)


  # # # # # # # # # # #
  # @ . . . . . o . . .
  # . # . # . # . # . #
  # . . . . . . . . . .
  # . # . # . # . # . #
  # . . . . . . . . . .
  # . # . # . # . # . #
  # . . . . . . . . . .
  # . # . # . # . # . #


## `field` -- the static layer

Only three values ever appear: `-1` wall, `0` free, `1` crate.
Bombs, coins, fire and agents are **not** in here -- they live in other keys.


In [4]:
print('shape', f.shape, 'dtype', f.dtype)
u, c = np.unique(f, return_counts=True)
print('values:', dict(zip(u.tolist(), c.tolist())))
print()
print('field[x, y] is [column, row] -- and y grows DOWNWARD:')
print('  field[1, 1] =', f[1,1], '(free)')
print('  field[2, 2] =', f[2,2], '(pillar)')
print('  field[0, 5] =', f[0,5], '(border)')
print()
for k, v in st.items():
    print(f'  {k:<14}', type(v).__name__, v if not isinstance(v, np.ndarray) else v.shape)


shape (17, 17) dtype int64
values: {-1: 113, 0: 176}

field[x, y] is [column, row] -- and y grows DOWNWARD:
  field[1, 1] = 0 (free)
  field[2, 2] = -1 (pillar)
  field[0, 5] = -1 (border)

  round          int 1
  step           int 10
  field          ndarray (17, 17)
  bombs          list []
  coins          list [(7, 1)]
  self           tuple ('me', 0, True, (1, 1))
  others         list []
  explosion_map  ndarray (17, 17)
  user_input     NoneType None


## `blast_coords` -- geometry only, no time

*If a bomb went off here, which tiles burn?* Walls stop an arm; crates do not.
Move the bomb and re-run to see arms get truncated.


In [5]:
BOMB = (3, 3)

g = empty_field()
g[3, 4] = 1                      # a crate in the DOWN arm
hit = blast_coords(g, *BOMB)

print(f'blast_coords(field, {BOMB[0]}, {BOMB[1]}) -> {len(hit)} tiles')
print(hit)
print()
draw(mk(g, BOMB), mark=hit)
print('\n  @ = the bomb, * = tiles that burn')
print('\n  note: the blast passes THROUGH the crate at (3,4) and keeps going.')

# Try these:
#   BOMB = (1, 1)   -> corner, two arms cut short, 7 tiles
#   BOMB = (5, 5)   -> wide open, the maximum 13 tiles


blast_coords(field, 3, 3) -> 11 tiles
[(3, 3), (3, 2), (3, 1), (4, 3), (5, 3), (6, 3), (3, 4), (3, 5), (3, 6), (2, 3), (1, 3)]

  # # # # # # # # # # #
  # . . * . . . . . . .
  # . # * # . # . # . #
  # * * @ * * * . . . .
  # . # * # . # . # . #
  # . . * . . . . . . .
  # . # * # . # . # . #
  # . . . . . . . . . .
  # . # . # . # . # . #

  @ = the bomb, * = tiles that burn

  note: the blast passes THROUGH the crate at (3,4) and keeps going.


## `bomb_deadlines` -- where time enters

For every tile: **at the end of which of my moves does it turn lethal?**
`LARGE` (999) means no bomb threatens it.

The key line is `deadline = timer + 1`.


In [6]:
st = mk(f, (1,1), bombs=[((3,3), 2)])
d = bomb_deadlines(st)

print('bomb at (3,3) with timer 2  ->  deadline', d[3,3])
print('a tile nothing threatens     ->', d[10,10], '(= LARGE)')
print()
print('deadline map (. = safe):')
for y in range(8):
    print('  ' + ' '.join('. ' if d[x,y] >= LARGE else f'{d[x,y]} ' for x in range(9)))
print()
print('every timer -> deadline:')
for t in range(s.BOMB_TIMER + 1):
    print(f'   timer {t}  ->  deadline {bomb_deadlines(mk(f,(1,1),bombs=[((3,3),t)]))[3,3]}')
print()
print('two overlapping bombs: the EARLIER deadline wins')
both = bomb_deadlines(mk(f, (1,1), bombs=[((3,3), 3), ((3,5), 0)]))
print('   tile (3,4) is in both blasts ->', both[3,4])


bomb at (3,3) with timer 2  ->  deadline 3
a tile nothing threatens     -> 999 (= LARGE)

deadline map (. = safe):
  .  .  .  .  .  .  .  .  . 
  .  .  .  3  .  .  .  .  . 
  .  .  .  3  .  .  .  .  . 
  .  3  3  3  3  3  3  .  . 
  .  .  .  3  .  .  .  .  . 
  .  .  .  3  .  .  .  .  . 
  .  .  .  3  .  .  .  .  . 
  .  .  .  .  .  .  .  .  . 

every timer -> deadline:
   timer 0  ->  deadline 1
   timer 1  ->  deadline 2
   timer 2  ->  deadline 3
   timer 3  ->  deadline 4
   timer 4  ->  deadline 5

two overlapping bombs: the EARLIER deadline wins
   tile (3,4) is in both blasts -> 1


## `is_lethal_at` -- the lingering fire

```python
d = deadlines[x, y]
return d <= move <= d + BLAST_LINGER
```

A tile is deadly at its deadline **and one move after**. Standing there a move later is fine.


In [7]:
st = mk(f, (1,1), bombs=[((3,3), 2)])
d = bomb_deadlines(st)
print(f'tile (3,3) has deadline {d[3,3]}\n')
for move in range(1, 8):
    hot = is_lethal_at(d, 3, 3, move)
    print(f'  standing there at the end of move {move}: '
          f'{"DEAD" if hot else "safe"}')


tile (3,3) has deadline 3

  standing there at the end of move 1: safe
  standing there at the end of move 2: safe
  standing there at the end of move 3: DEAD
  standing there at the end of move 4: DEAD
  standing there at the end of move 5: safe
  standing there at the end of move 6: safe
  standing there at the end of move 7: safe


## `can_survive` -- BFS over (position, *time*)

*Is there any escape route at all?* This is what makes `bomb_ok` trustworthy
rather than a guess. `extra_wait_first=True` models the BOMB action, which
spends move 1 standing still.


In [ ]:
def bomb_here(pos, field=None):
    field = empty_field() if field is None else field
    st = mk(field, pos)
    with_bomb = bomb_deadlines(st, extra_bomb=pos)
    ok = can_survive(st, pos, with_bomb, extra_wait_first=True)
    print(f'bomb at {pos}: {"SURVIVABLE" if ok else "SUICIDE"}')
    return ok

bomb_here((1,1))          # open corner -- fine
bomb_here((5,5))          # open crossroads -- fine

# Now wall ourselves into a dead end:
trap = empty_field()
trap[1,2] = 1; trap[2,1] = 1      # crates sealing the corner
print()
draw(mk(trap, (1,1)), w=6, h=5)
bomb_here((1,1), trap)    # no way out -> suicide


bomb at (1, 1): SURVIVABLE
bomb at (5, 5): SURVIVABLE

  # # # # # #
  # @ + . . .
  # + # . # .
  # . . . . .
  # . # . # .
bomb at (1, 1): SUICIDE


False

: 

## The BFS navigation helpers

`bfs_first_step` returns the **direction index** (0..3) of the first step toward the
nearest target -- this becomes `target_dir` after the `+1`. `bfs_to_safety` does the
same but aims at the nearest tile no bomb threatens.


In [ ]:
NAMES = ['UP', 'RIGHT', 'DOWN', 'LEFT']

st = mk(f, (1,1), coins=[(5,1)])
i = bfs_first_step(st, (1,1), st['coins'])
print(f'coin at (5,1): first step = {i} = {NAMES[i]}  ->  target_dir = {i+1}')

st = mk(f, (1,1), coins=[(1,5)])
i = bfs_first_step(st, (1,1), st['coins'])
print(f'coin at (1,5): first step = {i} = {NAMES[i]}  ->  target_dir = {i+1}')

print()
st = mk(f, (1,1), bombs=[((1,1), 3)])
d = bomb_deadlines(st)
i = bfs_to_safety(st, (1,1), d)
print(f'standing on our own bomb: escape = {i} = {NAMES[i]}')
draw(st, w=8, h=6)


## The two encodings, on the same board

This is the contrast your report is built on: nine numbers versus nine pictures.


In [ ]:
from agent_code.my_agent.callbacks import state_to_features as tabular
from agent_code.dqn_agent.features import (state_to_features as deep,
                                           CHANNEL_NAMES, SCALAR_NAMES)

scene = empty_field()
scene[3,1] = 1
st = mk(scene, (1,1), coins=[(7,1)], bombs=[((3,3), 2)],
        others=[('bob', 0, True, (5,3))])
draw(st)

names = ['target_dir','target_kind','tile_here','tile_up','tile_right',
         'tile_down','tile_left','bomb_ok','bomb_worth']
print('\nmy_agent  ->', tabular(st))
for n, v in zip(names, tabular(st)):
    print(f'   {n:<13} {v}')

board, scalars = deep(st)
print(f'\ndqn_agent -> board{board.shape} + scalars{scalars.shape}')
for i, n in enumerate(CHANNEL_NAMES):
    print(f'   plane {i} {n:<11} {int((board[i] > 0).sum()):>3} tiles lit')


## Scratch

Your turn. Build a board, ask a question, check the answer against the drawing.


In [ ]:
# e.g. where does a bomb at (5,5) reach, and could we survive dropping it?

pos = (5, 5)
board = empty_field()
st = mk(board, pos)

draw(st, mark=blast_coords(board, *pos))
print()
print('crates in blast   :', crates_hit(board, *pos))
print('opponents in blast:', opponents_hit(st, *pos))
print('survivable        :', can_survive(st, pos, bomb_deadlines(st, extra_bomb=pos),
                                          extra_wait_first=True))
